# RPG Eval-Seed Result Tables

This lightweight notebook loads eval-seed outputs from both the group artifacts tree and the repo-local home artifacts tree, then builds reporting tables for all available datasets and config tracks.

The eval script stays generic: it stores only per-user/per-seed metric rows and generic summaries. Paper targets and margin-specific checks such as relative ±5% mean matching and TOST equivalence are computed here, so changing the reporting criterion does not require rerunning Snellius jobs.

Expected result layout:

```text
<artifact-root>/rpg/eval_seeds/<track>/<dataset>/<session>/summary.json
<artifact-root>/rpg/eval_seeds/<track>/<dataset>/<session>/per_user_metrics.csv
```


In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

ROOT = Path.cwd() 
#  go back ..
while ROOT.name != "RPG":
    ROOT = ROOT.parent
HOME_ARTIFACT_ROOT = ROOT / "artifacts" / "rpg" / "eval_seeds"
GROUP_ARTIFACT_ROOT = Path("/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/eval_seeds")
ARTIFACT_ROOTS = []
for candidate in (GROUP_ARTIFACT_ROOT, HOME_ARTIFACT_ROOT):
    if candidate not in ARTIFACT_ROOTS:
        ARTIFACT_ROOTS.append(candidate)

REPORT_METRIC = "ndcg@10"
TABLE_METRICS = ["recall@5", "ndcg@5", "recall@10", "ndcg@10"]
METRIC_LABELS = {"recall@5": "R@5", "ndcg@5": "N@5", "recall@10": "R@10", "ndcg@10": "N@10"}
RELATIVE_EQ_MARGIN = 0.05  # 5% of the paper value.
TOST_CI_LEVEL = 0.90      # Standard two one-sided tests equivalence CI.
BOOTSTRAP_SAMPLES = 5000
BOOTSTRAP_SEED = 2024

PAPER_VALUES = {
    "recall@5": {
        "Sports": 0.0314,
        "Beauty": 0.0550,
        "Toys": 0.0592,
        "CDs": 0.0498,
    },
    "ndcg@5": {
        "Sports": 0.0216,
        "Beauty": 0.0381,
        "Toys": 0.0401,
        "CDs": 0.0338,
    },
    "recall@10": {
        "Sports": 0.0463,
        "Beauty": 0.0809,
        "Toys": 0.0869,
        "CDs": 0.0735,
    },
    "ndcg@10": {
        "Sports": 0.0263,
        "Beauty": 0.0464,
        "Toys": 0.0490,
        "CDs": 0.0415,
    },
}


def _format_scalar_for_latex(value: Any, formatter: Any, na_rep: str) -> Any:
    if pd.isna(value):
        return na_rep
    if formatter is None:
        return value
    if callable(formatter):
        return formatter(value)
    return formatter.format(value)


def latex_ready_frame(
    df: pd.DataFrame,
    formatter: Any = None,
    column_formatters: dict | None = None,
    na_rep: str = "",
) -> pd.DataFrame:
    formatted = df.copy()
    if formatter is not None:
        formatted = formatted.applymap(
            lambda value: _format_scalar_for_latex(value, formatter, na_rep)
        )
    if column_formatters:
        for column, column_formatter in column_formatters.items():
            formatted[column] = formatted[column].map(
                lambda value: _format_scalar_for_latex(value, column_formatter, na_rep)
            )
    return formatted


def emit_latex_table(
    name: str,
    df: pd.DataFrame,
    formatter: Any = None,
    column_formatters: dict | None = None,
    na_rep: str = "",
    escape: bool = True,
    index: bool = True,
) -> None:
    latex_df = latex_ready_frame(
        df,
        formatter=formatter,
        column_formatters=column_formatters,
        na_rep=na_rep,
    )
    print(f"\n{name} LaTeX:\n")
    print(
        latex_df.to_latex(
            index=index,
            escape=escape,
            na_rep=na_rep,
            multicolumn=True,
            multirow=True,
        )
    )

# If you run this notebook from another working directory, override one or both manually.
# HOME_ARTIFACT_ROOT = Path("/gpfs/home6/$USER/RPG/artifacts/rpg/eval_seeds").expanduser()
# GROUP_ARTIFACT_ROOT = Path("/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/eval_seeds")

ARTIFACT_ROOTS


[PosixPath('/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/eval_seeds'),
 PosixPath('/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds')]

In [2]:
TRACK_LABELS = {
    "released_readme": "released README",
    "paper_appendix": "paper appendix",
}

DATASET_LABELS = {
    "Sports_and_Outdoors": "Sports",
    "sports_and_outdoors": "Sports",
    "Beauty": "Beauty",
    "beauty": "Beauty",
    "Toys_and_Games": "Toys",
    "toys_and_games": "Toys",
    "CDs_and_Vinyl": "CDs",
    "cds_and_vinyl": "CDs",
}


def _metric_summary(payload: dict, metric: str) -> dict:
    for row in payload.get("metric_summary", []):
        if row.get("metric") == metric:
            return row
    raise KeyError(f"Metric {metric!r} not found in metric_summary")


def _paper_value(dataset: str, metric: str) -> float | None:
    value = PAPER_VALUES.get(metric, {}).get(dataset)
    return None if value is None else float(value)


def _mean_visited_items(payload: dict) -> float:
    values = [row.get("n_visited_items") for row in payload.get("per_seed_summary", [])]
    values = [float(value) for value in values if value is not None]
    return float(np.mean(values)) if values else math.nan


def _artifact_root_for(summary_path: Path) -> Path | None:
    for artifact_root in ARTIFACT_ROOTS:
        try:
            summary_path.relative_to(artifact_root)
            return artifact_root
        except ValueError:
            continue
    return None


def _track_and_dataset(summary_path: Path) -> tuple[str, str]:
    artifact_root = _artifact_root_for(summary_path)
    if artifact_root is None:
        return "unknown", "unknown"
    rel_parts = summary_path.relative_to(artifact_root).parts
    if len(rel_parts) >= 4 and rel_parts[0] in TRACK_LABELS:
        return rel_parts[0], rel_parts[1]
    if len(rel_parts) >= 3:
        return "unknown", rel_parts[0]
    return "unknown", "unknown"


def _bootstrap_ci(values: np.ndarray, *, n_samples: int, seed: int, ci_level: float, offset: float = 0.0) -> tuple[float, float]:
    values = np.asarray(values, dtype=np.float64)
    values = values[~np.isnan(values)]
    if values.size == 0:
        return math.nan, math.nan
    if n_samples <= 0:
        estimate = float(values.mean() - offset)
        return estimate, estimate

    rng = np.random.default_rng(seed)
    estimates = np.empty(n_samples, dtype=np.float64)
    for index in range(n_samples):
        sample_indices = rng.integers(0, values.shape[0], size=values.shape[0])
        estimates[index] = values[sample_indices].mean() - offset

    alpha = 1.0 - ci_level
    low, high = np.quantile(estimates, [alpha / 2.0, 1.0 - alpha / 2.0])
    return float(low), float(high)


def _per_user_seed_means(summary_path: Path, metric: str) -> np.ndarray | None:
    per_user_path = summary_path.parent / "per_user_metrics.csv"
    if not per_user_path.exists():
        return None
    per_user = pd.read_csv(per_user_path, usecols=["user_index", "eval_seed", metric])
    per_user[metric] = per_user[metric].astype(float)
    z_user = per_user.groupby("user_index", sort=True)[metric].mean()
    return z_user.to_numpy(dtype=np.float64)


def _relative_equivalence(summary_path: Path, metric: str, ours: float, paper: float | None) -> dict:
    if paper is None or paper == 0:
        return {
            "relative_margin_abs": math.nan,
            "relative_diff_pct": math.nan,
            "mean_within_relative_margin": None,
            "tost_diff_ci_low": math.nan,
            "tost_diff_ci_high": math.nan,
            "tost_equivalent_relative_margin": None,
            "tost_source": "missing paper value",
        }

    margin = abs(paper) * RELATIVE_EQ_MARGIN
    diff = ours - paper
    output = {
        "relative_margin_abs": margin,
        "relative_diff_pct": 100.0 * diff / paper,
        "mean_within_relative_margin": abs(diff) <= margin,
        "tost_diff_ci_low": math.nan,
        "tost_diff_ci_high": math.nan,
        "tost_equivalent_relative_margin": None,
        "tost_source": "missing per_user_metrics.csv",
    }

    z_user = _per_user_seed_means(summary_path, metric)
    if z_user is None:
        return output

    diff_low, diff_high = _bootstrap_ci(
        z_user,
        n_samples=BOOTSTRAP_SAMPLES,
        seed=BOOTSTRAP_SEED,
        ci_level=TOST_CI_LEVEL,
        offset=paper,
    )
    output.update(
        {
            "tost_diff_ci_low": diff_low,
            "tost_diff_ci_high": diff_high,
            "tost_equivalent_relative_margin": diff_low >= -margin and diff_high <= margin,
            "tost_source": "per_user_metrics.csv",
        }
    )
    return output


def load_eval_seed_runs(metric: str = REPORT_METRIC) -> pd.DataFrame:
    rows = []
    for artifact_root in ARTIFACT_ROOTS:
        if not artifact_root.exists():
            continue
        for summary_path in sorted(artifact_root.rglob("summary.json")):
            payload = json.loads(summary_path.read_text())
            track, dataset_slug = _track_and_dataset(summary_path)
            dataset = DATASET_LABELS.get(payload.get("category"), DATASET_LABELS.get(dataset_slug, dataset_slug))
            try:
                metric_row = _metric_summary(payload, metric=metric)
            except KeyError:
                continue
            paper = _paper_value(dataset, metric)
            ours = float(metric_row["final_user_avg"])
            diff = None if paper is None else ours - paper
            eval_seed_std = float(metric_row.get("eval_seed_std", math.nan))
            diff_over_seed_std = math.nan
            if diff is not None and eval_seed_std > 0:
                diff_over_seed_std = abs(diff) / eval_seed_std

            ci_low = float(metric_row.get("user_bootstrap_ci_low", math.nan))
            ci_high = float(metric_row.get("user_bootstrap_ci_high", math.nan))
            paper_inside_ci = None
            if paper is not None and not math.isnan(ci_low) and not math.isnan(ci_high):
                paper_inside_ci = ci_low <= paper <= ci_high

            rel_eq = _relative_equivalence(summary_path, metric, ours, paper)

            rows.append(
                {
                    "artifact_root": str(artifact_root),
                    "track": track,
                    "config_source": TRACK_LABELS.get(track, track),
                    "dataset_slug": dataset_slug,
                    "dataset": dataset,
                    "session": summary_path.parent.name,
                    "summary_path": str(summary_path),
                    "checkpoint_path": payload.get("checkpoint_path"),
                    "n_eval_seeds": len(payload.get("eval_seeds", [])),
                    "n_users": int(metric_row.get("n_users", 0)),
                    "metric": metric,
                    "paper_ndcg10": paper,
                    "ours_ndcg10": ours,
                    "diff": diff,
                    "eval_seed_std": eval_seed_std,
                    "diff_over_seed_std": diff_over_seed_std,
                    "user_ci_level": float(metric_row.get("user_bootstrap_ci_level", math.nan)),
                    "user_ci_low": ci_low,
                    "user_ci_high": ci_high,
                    "paper_inside_user_ci": paper_inside_ci,
                    "mean_visited_items": _mean_visited_items(payload),
                    "bootstrap_samples_summary": int(payload.get("bootstrap_samples", 0)),
                    "bootstrap_seed_summary": int(payload.get("bootstrap_seed", 0)),
                    **rel_eq,
                }
            )

    return pd.DataFrame(rows)


def load_all_metric_runs(metrics: list[str] = TABLE_METRICS) -> pd.DataFrame:
    frames = [load_eval_seed_runs(metric) for metric in metrics]
    frames = [frame for frame in frames if not frame.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


all_runs = load_eval_seed_runs(REPORT_METRIC)
all_metric_runs = load_all_metric_runs(TABLE_METRICS)
all_runs


,artifact_root,track,config_source,dataset_slug,dataset,session,summary_path,checkpoint_path,n_eval_seeds,n_users,metric,paper_ndcg10,ours_ndcg10,diff,eval_seed_std,diff_over_seed_std,user_ci_level,user_ci_low,user_ci_high,paper_inside_user_ci,mean_visited_items,bootstrap_samples_summary,bootstrap_seed_summary,relative_margin_abs,relative_diff_pct,mean_within_relative_margin,tost_diff_ci_low,tost_diff_ci_high,tost_equivalent_relative_margin,tost_source
0,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds,paper_appendix,paper appendix,beauty,Beauty,20260609T204607769709Z_job23617279,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...,/gpfs/work5/0/prjs2120/groups/group_16/artifac...,10,22363,ndcg@10,0.0464,0.043389,-0.003011,0.000278,10.824132,0.95,0.041278,0.045520,False,2120.079390,5000,2024,0.002320,-6.489955,False,-0.004788,-0.001205,False,per_user_metrics.csv
1,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds,paper_appendix,paper appendix,cds_and_vinyl,CDs,20260609T212008910885Z_job23617280,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...,/gpfs/work5/0/prjs2120/groups/group_16/artifac...,10,75258,ndcg@10,0.0415,0.037270,-0.004230,0.000169,25.022810,0.95,0.036232,0.038305,False,10195.944801,5000,2024,0.002075,-10.193702,False,-0.005086,-0.003361,False,per_user_metrics.csv
2,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds,paper_appendix,paper appendix,sports_and_outdoors,Sports,20260609T153816998982Z_job23610776,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...,/gpfs/home6/scur1202/RPG/artifacts/rpg/ckpt/rp...,10,35598,ndcg@10,0.0263,0.019454,-0.006846,0.000469,14.604141,0.95,0.018488,0.020461,False,1669.055127,5000,2024,0.001315,-26.031429,False,-0.007659,-0.006037,False,per_user_metrics.csv
3,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds,paper_appendix,paper appendix,sports_and_outdoors,Sports,20260609T204212482086Z_job23617224,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...,/gpfs/work5/0/prjs2120/groups/group_16/artifac...,10,35598,ndcg@10,0.0263,0.019454,-0.006846,0.000469,14.604141,0.95,0.018488,0.020461,False,1669.055127,5000,2024,0.001315,-26.031429,False,-0.007659,-0.006037,False,per_user_metrics.csv
4,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds,paper_appendix,paper appendix,toys_and_games,Toys,20260609T204626497279Z_job23617281,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...,/gpfs/work5/0/prjs2120/groups/group_16/artifac...,10,19412,ndcg@10,0.0490,0.044633,-0.004367,0.000318,13.739499,0.95,0.042454,0.046938,False,1184.159875,5000,2024,0.002450,-8.911382,False,-0.006234,-0.002461,False,per_user_metrics.csv
5,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds,released_readme,released README,beauty,Beauty,20260609T204950564140Z_job23617275,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...,/gpfs/work5/0/prjs2120/groups/group_16/artifac...,10,22363,ndcg@10,0.0464,0.045383,-0.001017,0.000106,9.567185,0.95,0.043164,0.047620,True,6202.643881,5000,2024,0.002320,-2.191029,True,-0.002907,0.000878,False,per_user_metrics.csv
6,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds,released_readme,released README,cds_and_vinyl,CDs,20260609T220243469378Z_job23617282,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...,/gpfs/work5/0/prjs2120/groups/group_16/artifac...,10,75258,ndcg@10,0.0415,0.040483,-0.001017,0.000196,5.188739,0.95,0.039345,0.041597,True,18856.181555,5000,2024,0.002075,-2.450400,True,-0.001976,-0.000084,True,per_user_metrics.csv
7,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds,released_readme,released README,sports_and_outdoors,Sports,20260609T154830898742Z_job23610777,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...,/gpfs/home6/scur1202/RPG/artifacts/rpg/ckpt/rp...,10,35598,ndcg@10,0.0263,0.025082,-0.001218,0.000136,8.959553,0.95,0.023725,0.026441,True,4573.590620,5000,2024,0.001315,-4.629748,True,-0.002363,-0.000067,False,per_user_metrics.csv
8,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds,released_readme,released README,sports_and_outdoors,Sports,20260609T205238919119Z_job23617226,/gpfs/home6/scur

In [3]:
if all_runs.empty:
    joined_roots = "\n".join(str(path) for path in ARTIFACT_ROOTS)
    print(f"No summary.json files found under:\n{joined_roots}")
else:
    # Keep the newest-looking session for each dataset/config source.
    latest = (
        all_runs.sort_values(["dataset", "track", "session"])
        .groupby(["dataset", "track"], as_index=False)
        .tail(1)
        .sort_values(["dataset", "track"])
        .reset_index(drop=True)
    )
    if all_metric_runs.empty:
        latest_all_metrics = pd.DataFrame()
    else:
        latest_sessions = latest[["dataset", "track", "session"]]
        latest_all_metrics = all_metric_runs.merge(latest_sessions, on=["dataset", "track", "session"], how="inner")
    latest_overview = latest[["dataset", "config_source", "session", "summary_path"]].copy()
    display(latest_overview)
    emit_latex_table("Latest runs", latest_overview)


,dataset,config_source,session,summary_path
0,Beauty,paper appendix,20260609T204607769709Z_job23617279,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...
1,Beauty,released README,20260609T204950564140Z_job23617275,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...
2,CDs,paper appendix,20260609T212008910885Z_job23617280,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...
3,CDs,released README,20260609T220243469378Z_job23617282,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...
4,Sports,paper appendix,20260609T204212482086Z_job23617224,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...
5,Sports,released README,20260609T205238919119Z_job23617226,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...
6,Toys,paper appendix,20260609T204626497279Z_job23617281,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...
7,Toys,released README,20260609T204759294488Z_job23617278,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_se...



Latest runs LaTeX:

\begin{tabular}{lllll}
\toprule
 & dataset & config\_source & session & summary\_path \\
\midrule
0 & Beauty & paper appendix & 20260609T204607769709Z\_job23617279 & /gpfs/home6/scur1202/RPG/artifacts/rpg/eval\_seeds/paper\_appendix/beauty/20260609T204607769709Z\_job23617279/summary.json \\
1 & Beauty & released README & 20260609T204950564140Z\_job23617275 & /gpfs/home6/scur1202/RPG/artifacts/rpg/eval\_seeds/released\_readme/beauty/20260609T204950564140Z\_job23617275/summary.json \\
2 & CDs & paper appendix & 20260609T212008910885Z\_job23617280 & /gpfs/home6/scur1202/RPG/artifacts/rpg/eval\_seeds/paper\_appendix/cds\_and\_vinyl/20260609T212008910885Z\_job23617280/summary.json \\
3 & CDs & released README & 20260609T220243469378Z\_job23617282 & /gpfs/home6/scur1202/RPG/artifacts/rpg/eval\_seeds/released\_readme/cds\_and\_vinyl/20260609T220243469378Z\_job23617282/summary.json \\
4 & Sports & paper appendix & 20260609T204212482086Z\_job23617224 & /gpfs/home6/scur1202/

## Table 1: Our Results

This table mirrors the paper's Table 2 layout, but reports only our runs. Values are `final_user_avg`: average each user over eval seeds, then average over users.

In [4]:
if not all_metric_runs.empty and not latest_all_metrics.empty:
    dataset_order = ["Sports", "Beauty", "Toys", "CDs"]
    metric_order = TABLE_METRICS

    our_results = latest_all_metrics[
        ["config_source", "dataset", "metric", "ours_ndcg10"]
    ].copy()
    our_results["metric_label"] = our_results["metric"].map(METRIC_LABELS)
    our_results["Run"] = our_results["config_source"]

    paper_rows = []
    for dataset in dataset_order:
        for metric in metric_order:
            paper_value = PAPER_VALUES.get(metric, {}).get(dataset)
            if paper_value is None:
                continue
            paper_rows.append(
                {
                    "config_source": "paper",
                    "dataset": dataset,
                    "metric": metric,
                    "ours_ndcg10": paper_value,
                    "metric_label": METRIC_LABELS[metric],
                    "Run": "paper",
                }
            )
    if paper_rows:
        our_results = pd.concat([our_results, pd.DataFrame(paper_rows)], ignore_index=True)

    table_ours = our_results.pivot_table(
        index="Run",
        columns=["dataset", "metric_label"],
        values="ours_ndcg10",
        aggfunc="first",
    )

    ordered_columns = [
        (dataset, METRIC_LABELS[metric])
        for dataset in dataset_order
        for metric in metric_order
        if (dataset, METRIC_LABELS[metric]) in table_ours.columns
    ]
    ordered_index = [label for label in ["paper", "paper appendix", "released README"] if label in table_ours.index]
    table_ours = table_ours.loc[ordered_index, ordered_columns]
    table_ours.index.name = "Model/config"

    display(table_ours.style.format("{:.4f}", na_rep=""))
    emit_latex_table("Table 1", table_ours, formatter="{:.4f}", na_rep="", escape=True)



Table 1 LaTeX:

\begin{tabular}{lllllllllllllllll}
\toprule
dataset & \multicolumn{4}{r}{Sports} & \multicolumn{4}{r}{Beauty} & \multicolumn{4}{r}{Toys} & \multicolumn{4}{r}{CDs} \\
metric_label & R@5 & N@5 & R@10 & N@10 & R@5 & N@5 & R@10 & N@10 & R@5 & N@5 & R@10 & N@10 & R@5 & N@5 & R@10 & N@10 \\
Model/config &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
paper & 0.0314 & 0.0216 & 0.0463 & 0.0263 & 0.0550 & 0.0381 & 0.0809 & 0.0464 & 0.0592 & 0.0401 & 0.0869 & 0.0490 & 0.0498 & 0.0338 & 0.0735 & 0.0415 \\
paper appendix & 0.0230 & 0.0161 & 0.0333 & 0.0195 & 0.0520 & 0.0361 & 0.0747 & 0.0434 & 0.0532 & 0.0362 & 0.0794 & 0.0446 & 0.0453 & 0.0309 & 0.0651 & 0.0373 \\
released README & 0.0293 & 0.0205 & 0.0435 & 0.0251 & 0.0539 & 0.0375 & 0.0782 & 0.0454 & 0.0572 & 0.0389 & 0.0851 & 0.0478 & 0.0490 & 0.0332 & 0.0715 & 0.0405 \\
\bottomrule
\end{tabular}



/scratch-local/63462/ipykernel_502050/3868764676.py:78: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted = formatted.applymap(


## Table 2: NDCG@10 Paper Comparison

This table keeps the formal comparison focused on the primary endpoint, `NDCG@10`. The relative margin and TOST confidence level are controlled in the first notebook cell.

In [5]:
if not all_runs.empty:
    table_reproduction = latest[
        [
            "dataset",
            "config_source",
            "paper_ndcg10",
            "ours_ndcg10",
            "diff",
            "relative_diff_pct",
            "relative_margin_abs",
            "tost_diff_ci_low",
            "tost_diff_ci_high",
            "mean_within_relative_margin",
            "tost_equivalent_relative_margin",
            "eval_seed_std",
            "diff_over_seed_std",
        ]
    ].copy()
    table_reproduction[f"{TOST_CI_LEVEL:.0%} TOST diff CI"] = table_reproduction.apply(
        lambda row: "missing" if pd.isna(row["tost_diff_ci_low"]) else f"[{row['tost_diff_ci_low']:+.5f}, {row['tost_diff_ci_high']:+.5f}]",
        axis=1,
    )
    table_reproduction = table_reproduction[
        [
            "dataset",
            "config_source",
            "paper_ndcg10",
            "ours_ndcg10",
            "diff",
            "relative_diff_pct",
            "relative_margin_abs",
            f"{TOST_CI_LEVEL:.0%} TOST diff CI",
            "mean_within_relative_margin",
            "tost_equivalent_relative_margin",
            "eval_seed_std",
            "diff_over_seed_std",
        ]
    ].rename(
        columns={
            "dataset": "Dataset",
            "config_source": "Config source",
            "paper_ndcg10": "Paper NDCG@10",
            "ours_ndcg10": "Ours NDCG@10",
            "diff": "Diff",
            "relative_diff_pct": "Rel. diff %",
            "relative_margin_abs": f"±{RELATIVE_EQ_MARGIN:.0%} margin",
            "mean_within_relative_margin": f"Mean within ±{RELATIVE_EQ_MARGIN:.0%}?",
            "tost_equivalent_relative_margin": f"TOST equivalent ±{RELATIVE_EQ_MARGIN:.0%}?",
            "eval_seed_std": "Eval seed std",
            "diff_over_seed_std": "|Diff| / seed std",
        }
    )
    table_reproduction_formatters = {
        "Paper NDCG@10": "{:.4f}",
        "Ours NDCG@10": "{:.5f}",
        "Diff": "{:+.5f}",
        "Rel. diff %": "{:+.1f}%",
        f"±{RELATIVE_EQ_MARGIN:.0%} margin": "{:.5f}",
        "Eval seed std": "{:.6f}",
        "|Diff| / seed std": "{:.1f}",
    }
    display(table_reproduction.style.format({
        "Paper NDCG@10": "{:.4f}",
        "Ours NDCG@10": "{:.5f}",
        "Diff": "{:+.5f}",
        "Rel. diff %": "{:+.1f}%",
        f"±{RELATIVE_EQ_MARGIN:.0%} margin": "{:.5f}",
        "Eval seed std": "{:.6f}",
        "|Diff| / seed std": "{:.1f}",
    }))
    emit_latex_table("Table 2", table_reproduction, column_formatters=table_reproduction_formatters)


,Dataset,Config source,Paper NDCG@10,Ours NDCG@10,Diff,Rel. diff %,±5% margin,90% TOST diff CI,Mean within ±5%?,TOST equivalent ±5%?,Eval seed std,|Diff| / seed std
0,Beauty,paper appendix,0.0464,0.04339,-0.00301,-6.5%,0.00232,"[-0.00479, -0.00120]",False,False,0.000278,10.8
1,Beauty,released README,0.0464,0.04538,-0.00102,-2.2%,0.00232,"[-0.00291, +0.00088]",True,False,0.000106,9.6
2,CDs,paper appendix,0.0415,0.03727,-0.00423,-10.2%,0.00208,"[-0.00509, -0.00336]",False,False,0.000169,25.0
3,CDs,released README,0.0415,0.04048,-0.00102,-2.5%,0.00208,"[-0.00198, -0.00008]",True,True,0.000196,5.2
4,Sports,paper appendix,0.0263,0.01945,-0.00685,-26.0%,0.00132,"[-0.00766, -0.00604]",False,False,0.000469,14.6
5,Sports,released README,0.0263,0.02508,-0.00122,-4.6%,0.00132,"[-0.00236, -0.00007]",True,False,0.000136,9.0
6,Toys,paper appendix,0.0490,0.04463,-0.00437,-8.9%,0.00245,"[-0.00623, -0.00246]",False,False,0.000318,13.7
7,Toys,released README,0.0490,0.04785,-0.00115,-2.4%,0.00245,"[-0.00314, +0.00096]",True,False,0.000177,6.5



Table 2 LaTeX:

\begin{tabular}{lllllllllrrll}
\toprule
 & Dataset & Config source & Paper NDCG@10 & Ours NDCG@10 & Diff & Rel. diff \% & ±5\% margin & 90\% TOST diff CI & Mean within ±5\%? & TOST equivalent ±5\%? & Eval seed std & |Diff| / seed std \\
\midrule
0 & Beauty & paper appendix & 0.0464 & 0.04339 & -0.00301 & -6.5\% & 0.00232 & [-0.00479, -0.00120] & False & False & 0.000278 & 10.8 \\
1 & Beauty & released README & 0.0464 & 0.04538 & -0.00102 & -2.2\% & 0.00232 & [-0.00291, +0.00088] & True & False & 0.000106 & 9.6 \\
2 & CDs & paper appendix & 0.0415 & 0.03727 & -0.00423 & -10.2\% & 0.00208 & [-0.00509, -0.00336] & False & False & 0.000169 & 25.0 \\
3 & CDs & released README & 0.0415 & 0.04048 & -0.00102 & -2.5\% & 0.00208 & [-0.00198, -0.00008] & True & True & 0.000196 & 5.2 \\
4 & Sports & paper appendix & 0.0263 & 0.01945 & -0.00685 & -26.0\% & 0.00132 & [-0.00766, -0.00604] & False & False & 0.000469 & 14.6 \\
5 & Sports & released README & 0.0263 & 0.02508 & -0.00122 

## Table 3: Bootstrap CI

This table reports the classical IR-style uncertainty over users for `NDCG@10`. It should be interpreted separately from eval-seed variance and the equivalence margin.

In [6]:
if not all_runs.empty:
    table_user_ci = latest[
        [
            "dataset",
            "config_source",
            "ours_ndcg10",
            "user_ci_low",
            "user_ci_high",
            "paper_inside_user_ci",
            "bootstrap_samples_summary",
            "bootstrap_seed_summary",
        ]
    ].copy()
    table_user_ci["User bootstrap CI"] = table_user_ci.apply(
        lambda row: f"[{row['user_ci_low']:.5f}, {row['user_ci_high']:.5f}]",
        axis=1,
    )
    table_user_ci = table_user_ci[
        ["dataset", "config_source", "ours_ndcg10", "User bootstrap CI", "paper_inside_user_ci", "bootstrap_samples_summary", "bootstrap_seed_summary"]
    ].rename(
        columns={
            "dataset": "Dataset",
            "config_source": "Config source",
            "ours_ndcg10": "Ours NDCG@10",
            "paper_inside_user_ci": "Paper inside user CI?",
            "bootstrap_samples_summary": "Summary bootstrap samples",
            "bootstrap_seed_summary": "Summary bootstrap seed",
        }
    )
    display(table_user_ci.style.format({"Ours NDCG@10": "{:.5f}"}))
    emit_latex_table("Table 3", table_user_ci, column_formatters={"Ours NDCG@10": "{:.5f}"})


,Dataset,Config source,Ours NDCG@10,User bootstrap CI,Paper inside user CI?,Summary bootstrap samples,Summary bootstrap seed
0,Beauty,paper appendix,0.04339,"[0.04128, 0.04552]",False,5000,2024
1,Beauty,released README,0.04538,"[0.04316, 0.04762]",True,5000,2024
2,CDs,paper appendix,0.03727,"[0.03623, 0.03830]",False,5000,2024
3,CDs,released README,0.04048,"[0.03935, 0.04160]",True,5000,2024
4,Sports,paper appendix,0.01945,"[0.01849, 0.02046]",False,5000,2024
5,Sports,released README,0.02508,"[0.02372, 0.02644]",True,5000,2024
6,Toys,paper appendix,0.04463,"[0.04245, 0.04694]",False,5000,2024
7,Toys,released README,0.04785,"[0.04543, 0.05036]",True,5000,2024



Table 3 LaTeX:

\begin{tabular}{lllllrrr}
\toprule
 & Dataset & Config source & Ours NDCG@10 & User bootstrap CI & Paper inside user CI? & Summary bootstrap samples & Summary bootstrap seed \\
\midrule
0 & Beauty & paper appendix & 0.04339 & [0.04128, 0.04552] & False & 5000 & 2024 \\
1 & Beauty & released README & 0.04538 & [0.04316, 0.04762] & True & 5000 & 2024 \\
2 & CDs & paper appendix & 0.03727 & [0.03623, 0.03830] & False & 5000 & 2024 \\
3 & CDs & released README & 0.04048 & [0.03935, 0.04160] & True & 5000 & 2024 \\
4 & Sports & paper appendix & 0.01945 & [0.01849, 0.02046] & False & 5000 & 2024 \\
5 & Sports & released README & 0.02508 & [0.02372, 0.02644] & True & 5000 & 2024 \\
6 & Toys & paper appendix & 0.04463 & [0.04245, 0.04694] & False & 5000 & 2024 \\
7 & Toys & released README & 0.04785 & [0.04543, 0.05036] & True & 5000 & 2024 \\
\bottomrule
\end{tabular}



## Appendix Table: Full Run Metadata

Useful for checking which checkpoint/session produced each row.


In [7]:
if not all_runs.empty:
    metadata_cols = [
        "dataset",
        "config_source",
        "session",
        "n_eval_seeds",
        "n_users",
        "mean_visited_items",
        "checkpoint_path",
        "summary_path",
    ]
    metadata_table = latest[metadata_cols].copy()
    display(metadata_table.style.format({"mean_visited_items": "{:.1f}"}))
    emit_latex_table("Appendix Table", metadata_table, column_formatters={"mean_visited_items": "{:.1f}"})


,dataset,config_source,session,n_eval_seeds,n_users,mean_visited_items,checkpoint_path,summary_path
0,Beauty,paper appendix,20260609T204607769709Z_job23617279,10,22363,2120.1,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/ckpt/rpg_repro_beauty-scripts|rpg.py_--preset_beauty-Jun-05-2026_15-10-e7b432.pth,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds/paper_appendix/beauty/20260609T204607769709Z_job23617279/summary.json
1,Beauty,released README,20260609T204950564140Z_job23617275,10,22363,6202.6,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/ckpt/rpg_repro_beauty-scripts|rpg.py_--preset_beauty-Jun-05-2026_15-10-e7b432.pth,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds/released_readme/beauty/20260609T204950564140Z_job23617275/summary.json
2,CDs,paper appendix,20260609T212008910885Z_job23617280,10,75258,10195.9,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/ckpt/rpg_repro_cds_and_vinyl-scripts|rpg.py_--preset_cds_and_vinyl-Jun-05-2026_17-17-07ec7f.pth,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds/paper_appendix/cds_and_vinyl/20260609T212008910885Z_job23617280/summary.json
3,CDs,released README,20260609T220243469378Z_job23617282,10,75258,18856.2,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/ckpt/rpg_repro_cds_and_vinyl-scripts|rpg.py_--preset_cds_and_vinyl-Jun-05-2026_17-17-07ec7f.pth,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds/released_readme/cds_and_vinyl/20260609T220243469378Z_job23617282/summary.json
4,Sports,paper appendix,20260609T204212482086Z_job23617224,10,35598,1669.1,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/ckpt/rpg_repro_sports_and_outdoors-scripts|rpg.py_--preset_sports_and_outdoors-Jun-05-2026_17-18-c4d3a6.pth,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds/paper_appendix/sports_and_outdoors/20260609T204212482086Z_job23617224/summary.json
5,Sports,released README,20260609T205238919119Z_job23617226,10,35598,4573.6,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/ckpt/rpg_repro_sports_and_outdoors-scripts|rpg.py_--preset_sports_and_outdoors-Jun-05-2026_17-18-c4d3a6.pth,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds/released_readme/sports_and_outdoors/20260609T205238919119Z_job23617226/summary.json
6,Toys,paper appendix,20260609T204626497279Z_job23617281,10,19412,1184.2,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/ckpt/rpg_repro_toys_and_games-scripts|rpg.py_--preset_toys_and_games-Jun-05-2026_17-17-1e5fbf.pth,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds/paper_appendix/toys_and_games/20260609T204626497279Z_job23617281/summary.json
7,Toys,released README,20260609T204759294488Z_job23617278,10,19412,5152.2,/gpfs/work5/0/prjs2120/groups/group_16/artifacts/rpg/ckpt/rpg_repro_toys_and_games-scripts|rpg.py_--preset_toys_and_games-Jun-05-2026_17-17-1e5fbf.pth,/gpfs/home6/scur1202/RPG/artifacts/rpg/eval_seeds/released_readme/toys_and_games/20260609T204759294488Z_job23617278/summary.json



Appendix Table LaTeX:

\begin{tabular}{llllrrlll}
\toprule
 & dataset & config\_source & session & n\_eval\_seeds & n\_users & mean\_visited\_items & checkpoint\_path & summary\_path \\
\midrule
0 & Beauty & paper appendix & 20260609T204607769709Z\_job23617279 & 10 & 22363 & 2120.1 & /gpfs/work5/0/prjs2120/groups/group\_16/artifacts/rpg/ckpt/rpg\_repro\_beauty-scripts|rpg.py\_--preset\_beauty-Jun-05-2026\_15-10-e7b432.pth & /gpfs/home6/scur1202/RPG/artifacts/rpg/eval\_seeds/paper\_appendix/beauty/20260609T204607769709Z\_job23617279/summary.json \\
1 & Beauty & released README & 20260609T204950564140Z\_job23617275 & 10 & 22363 & 6202.6 & /gpfs/work5/0/prjs2120/groups/group\_16/artifacts/rpg/ckpt/rpg\_repro\_beauty-scripts|rpg.py\_--preset\_beauty-Jun-05-2026\_15-10-e7b432.pth & /gpfs/home6/scur1202/RPG/artifacts/rpg/eval\_seeds/released\_readme/beauty/20260609T204950564140Z\_job23617275/summary.json \\
2 & CDs & paper appendix & 20260609T212008910885Z\_job23617280 & 10 & 75258 & 10195.